In [1]:
import pandas as pd
import xarray as xr
import xcdat as xc
import numpy as np
import os
import glob
import re
import cftime

/global/homes/j/jungchoi/.conda/envs/pcmdi_metrics/lib/python3.10/site-packages/esmpy/interface/loadESMF.py:94: VersionWarning: ESMF installation version 8.8.0, ESMPy version 8.8.0b0
  warnings.warn("ESMF installation version {}, ESMPy version {}".format(


Model information, Target period

In [2]:
print("%%%%%%%% start %%%%%%%")
output_dir = "/pscratch/sd/j/jungchoi/DCPP/_metrics"
output_grid = xc.regridder.grid.create_uniform_grid(-88.75, 88.75, 2.5, 0.0, 357.5, 2.5)
output_grid_no = "144x72"

input_dir = "/pscratch/sd/j/jungchoi/DCPP/_link"
experiment = "historical"


target_year_start = 1961
target_year_end = 2014

%%%%%%%% start %%%%%%%


# Targetyear average, Horizonal interpolarion for each ensemble members 

In [3]:
#mdl_var_name = "tas"
mdl_var_name = "pr"
mdl_list = ['CanESM5', 'CMCC-CM2-SR5', 'CNRM-ESM2-1', 'EC-Earth3', 'FGOALS-f3-L', 'HadGEM3-GC31-MM', 'IPSL-CM6A-LR', 'MIROC6', 'MPI-ESM1-2-HR', 'MRI-ESM2-0', 'NorCPM1']

for model in mdl_list[4:5]:
    ensemble_no = 10
    if( model == "FGOALS-f3-L" ):
        ensemble_no = 3
    if( model == "HadGEM3-GC31-MM" ):
        ensemble_no = 4

    ensemble_ds = []

    for ensemble in range(1, ensemble_no + 1):
        file_pattern = os.path.join(input_dir, model, experiment, f"{mdl_var_name}_r{ensemble}_*.nc")
        file_list = sorted(glob.glob(file_pattern))  # Get all matching files
              
        nfile = len(file_list)            
        selected_file = []
        #print(nfile, file_list)

        all_annual_mean = []
        if nfile == 1: 
            selected_file.append(file_list[0])
            ds = xr.open_dataset(selected_file[0])
            time_values = ds["time"].values
            ds_selected = ds.where((ds.time.dt.year >= target_year_start) & (ds.time.dt.year <= target_year_end),drop=True)
            #print(ds_selected.time)
            ds_annual_mean = ds_selected.groupby('time.year').mean(dim="time")  # Average over time dimension
            ds.close()
            all_annual_mean = ds_annual_mean
            #print('nfile=1', all_annual_mean)
    
        if nfile > 1:
            for file in file_list:
                period = file.split("_")[-1]
                start_str, end_str = period.split("-")
    
                start_year = int(start_str[:4])
                end_year = int(end_str[:4])

                #print(file, start_year, end_year)
                if not (end_year < target_year_start or start_year > target_year_end):
                    selected_file.append(file)
            #print(selected_file)
     
            for file in selected_file:       
                ds = xr.open_dataset(file)
                time_values = ds["time"].values
                ds_selected = ds.where((ds.time.dt.year >= target_year_start) & (ds.time.dt.year <= target_year_end),drop=True)
                ds_annual_mean = ds_selected.groupby('time.year').mean(dim="time")  # Average over time dimension
                ds.close()
                all_annual_mean.append(ds_annual_mean)
            
            all_annual_mean = xr.concat(all_annual_mean, dim="year")
            #print('nfile>1', all_annual_mean)

        print(model, ensemble, type(all_annual_mean))
        all_annual_mean = all_annual_mean.rename({'year': 'time'})
        years = all_annual_mean['time'].values
        all_annual_mean['time'] = pd.to_datetime([f'{y}-01-01' for y in years])
        #print(all_annual_mean['time'].values)
        
        #print(all_annual_mean)
        #print(all_annual_mean.coords)

        if "lat_bnds" in all_annual_mean.variables and "lon_bnds" in all_annual_mean.variables:
            all_annual_mean["lat_bnds"] = all_annual_mean["lat_bnds"].isel(time=0)
            all_annual_mean["lon_bnds"] = all_annual_mean["lon_bnds"].isel(time=0)
            all_annual_mean = all_annual_mean.set_coords(["lat_bnds", "lon_bnds"])
        else:
            all_annual_mean['lat'].attrs['axis'] = 'Y'
            all_annual_mean['lon'].attrs['axis'] = 'X'
            all_annual_mean = all_annual_mean.bounds.add_bounds("Y")
            all_annual_mean = all_annual_mean.bounds.add_bounds("X")
    
        regrid_ds = all_annual_mean.regridder.horizontal(f"{mdl_var_name}", output_grid, tool="regrid2")
        ensemble_ds.append(regrid_ds)
    
    all_ensemble_ds = xr.concat(ensemble_ds, dim="ensemble")
    ##all_ensemble_ds = all_ensemble_ds.mean(dim="ensemble")
    print(all_ensemble_ds)

    if mdl_var_name == "pr":
        all_ensemble_ds = all_ensemble_ds * 60 * 60 * 24
    
    # Save to NetCDF for each lead time
    output_filename = f"{output_dir}/{model}/{mdl_var_name}.{output_grid_no}.{experiment}.{target_year_start}-{target_year_end}.r1-{ensemble_no}.nc"
    if os.path.exists(output_filename):
        os.remove(output_filename)
    all_ensemble_ds.to_netcdf(output_filename)
    print(f"%% Regrid time-averaged dataset saved: {output_filename}")

FGOALS-f3-L 1 <class 'xarray.core.dataset.Dataset'>
FGOALS-f3-L 2 <class 'xarray.core.dataset.Dataset'>
FGOALS-f3-L 3 <class 'xarray.core.dataset.Dataset'>
<xarray.Dataset> Size: 7MB
Dimensions:   (ensemble: 3, time: 54, lat: 72, lon: 144, bnds: 2)
Coordinates:
  * time      (time) datetime64[ns] 432B 1961-01-01 1962-01-01 ... 2014-01-01
  * lat       (lat) float64 576B -88.75 -86.25 -83.75 ... 83.75 86.25 88.75
  * lon       (lon) float64 1kB 0.0 2.5 5.0 7.5 10.0 ... 350.0 352.5 355.0 357.5
Dimensions without coordinates: ensemble, bnds
Data variables:
    pr        (ensemble, time, lat, lon) float32 7MB 2.397e-06 ... 6.59e-06
    lon_bnds  (ensemble, lon, bnds) float64 7kB -1.25 1.25 1.25 ... 356.2 358.8
    lat_bnds  (ensemble, lat, bnds) float64 3kB -90.0 -87.5 -87.5 ... 87.5 90.0
Attributes: (12/46)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            CMIP
    branch_method:          Spin-up documentation
    branch_time_in_child:   2310.0
    branch_time_in_par

In [4]:
import matplotlib.pyplot as plt

time = all_ensemble_ds.time.values
n_ens = all_ensemble_ds.sizes["ensemble"]  # 또는 n_ens = all_ensemble_ds.ensemble.size

plt.figure(figsize=(12, 6))

for ens in range(n_ens):
    tas_series = all_ensemble_ds.tas[ens, :, 36, 72].values
    plt.plot(time, tas_series, label=f"Ens {ens}")

plt.title("Temperature Time Series at (lat=36, lon=72) - All Ensembles")
plt.xlabel("Time")
plt.ylabel("Temperature (tas)")
plt.legend(ncol=2, fontsize="small")
plt.grid(True)
plt.tight_layout()
plt.show()
 

AttributeError: 'Dataset' object has no attribute 'tas'

<Figure size 1200x600 with 0 Axes>